[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/42.%20The%20Peak%20Energy%20Consumption%20%26%20Load%20Shifting%20Problem/P42-Tier-3_executed.ipynb)

---


# 42. The Peak Energy Consumption & Load Shifting Problem

## Tier 3: Moth-Flame Optimization

### Goal

Use the Moth-Flame Optimization (MFO) metaheuristic to search for a lower-cost daily schedule for a set of energy-consuming operations. Each moth represents one candidate schedule; flames represent the strongest candidate schedules in the current population. Moths move relative to flames using a spiral update, and the number of flames decreases as the search proceeds.

MFO is a stochastic search method, not a proof of global optimality. Its output is a candidate solution whose quality depends on the problem, parameter settings, and random initialization.

### Inputs and outputs

The input is a 24-hour energy-scheduling instance: operation durations and energy use, hourly electricity prices, and a baseline demand profile. The optimizer returns integer start times and the associated daily cost. It also records best-cost, average-cost, and population-diversity histories for analysis.

### Model assumptions and scope

- Each operation is scheduled once, starts at an integer hour, and runs continuously for its specified duration.
- The operation's total energy is distributed uniformly across its operating hours, so its hourly power contribution is constant.
- The specified operation windows cover the full 24-hour horizon. No precedence, resource-capacity, or mutual-exclusion constraints are modeled.
- The cost includes hourly energy charges and a demand charge applied to the maximum hourly demand. The storage parameters are provided as instance data but storage dispatch is not part of this scheduling model.
- Operation start times are represented continuously during MFO movement, then rounded and clipped to feasible integer start times when a candidate is evaluated.

### Mathematical model

Let \(\mathcal{O}\) be the set of operations and \(\mathcal{T}=\{1,\ldots,24\}\) the set of hourly periods. For operation \(i\), let \(d_i\) be its duration, \(E_i\) its energy use in kWh, and \(s_i\) its integer start hour. Its hourly power while operating is \(E_i/d_i\) kW. Let \(B_t\) be baseline demand in MW and \(p_t\) the energy price in dollars per kWh. The hourly demand is

$$
D_t(s)=B_t+\frac{1}{1000}\sum_{i\in\mathcal{O}}\frac{E_i}{d_i}\,\mathbf{1}\{s_i\le t\le s_i+d_i-1\},\qquad t\in\mathcal{T}.
$$

The objective is to minimize energy charges plus a demand charge:

$$
C(s)=\sum_{t\in\mathcal{T}}p_t\,(1000D_t(s)) + q\,(1000\max_{t\in\mathcal{T}}D_t(s)),
$$

subject to

$$
s_i\in\{1,\ldots,25-d_i\},\qquad i\in\mathcal{O}.
$$

Here, \(\mathbf{1}\{\cdot\}\) equals 1 when the operation is running in hour \(t\), and 0 otherwise; \(q\) is the demand-charge coefficient in dollars per kW. Multiplication by 1000 converts MW to kW, and for a one-hour period the numerical energy in kWh equals the average power in kW. In the code, these quantities are represented by `problem.operations`, `problem.base_demand`, `problem.prices`, and `problem.demand_charge`; `calculate_hourly_demand` and `calculate_solution_cost` implement the demand profile and objective.

### MFO search and discrete schedules

A continuous vector encodes the start times, with one coordinate per operation. Each coordinate is normalized to \([0,1]\), then decoded to an integer hour before evaluating the schedule. This allows the continuous spiral update to search over a representation of a discrete scheduling problem.

For moth \(i\) and its assigned flame \(j\), the component-wise distance and spiral update are

$$
D_i=|F_j-M_i|,\qquad
S(M_i,F_j)=D_i\,e^{bt}\cos(2\pi t)+F_j.
$$

Here, \(M_i\) is the moth position, \(F_j\) is the flame position, \(b\) is the spiral constant, and \(t\) is a parameter that changes from \(-1\) to \(1\) over the iterations. The absolute value and other operations are applied coordinate by coordinate. After each update, coordinates are clipped to \([0,1]\). The implementation is in `spiral_function`, with the coordinate-wise distance formed in `run_mfo`.

The algorithm initializes a population, ranks moths by fitness, selects the leading moths as flames, moves the moths, and repeats. Fitness is defined as \(f(s)=1/(1+C(s))\), so a lower nonnegative cost gives a higher fitness. The flame count decreases linearly, with at least one flame retained. The recorded best cost is the best encountered during the run; the average cost describes the current population, not the best-so-far value.

For population size \(P\), dimension \(|\mathcal{O}|\), and \(K\) iterations, the population-based search performs roughly \(O(KP|\mathcal{O}|)\) candidate-coordinate updates, in addition to cost evaluations and diversity calculations. This is a heuristic search: it does not certify that the returned schedule is globally optimal.

### What to examine

- Whether the best encountered cost decreases and then levels off.
- How the population's average cost differs from its best cost.
- How pairwise population diversity changes as the flame count shrinks.
- How the optimized hourly demand profile compares with the all-operations-at-hour-1 baseline.
- How much energy is actually scheduled during peak-price and off-peak-price hours. An operation may span both price periods.

The instance contains 23 operations. The population size, iteration limit, and spiral constant used in the example are specified when the optimizer is created. Results are generated by running the notebook; no particular savings percentage or convergence iteration is guaranteed.

In [ ]:
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.spatial.distance import euclidean

# Fixed seeds make the random initialization repeatable when cells run in order.
np.random.seed(42)
random.seed(42)

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
# Define the energy-scheduling instance used by the MFO demonstration.
class MFOEnergyProblem:
    def __init__(self):
        self.T = list(range(1, 25))

        # Each operation has a duration in hours, total energy in kWh,
        # and an allowed start-time window.
        self.operations = {
            "container_crane_a": {"duration": 4, "energy": 350, "window": list(range(1, 25))},
            "container_crane_b": {"duration": 4, "energy": 350, "window": list(range(1, 25))},
            "refrigeration_main": {"duration": 8, "energy": 420, "window": list(range(1, 25))},
            "ev_charging_hub": {"duration": 6, "energy": 480, "window": list(range(1, 25))},
            "conveyor_main": {"duration": 5, "energy": 380, "window": list(range(1, 25))},
            "warehouse_hvac_1": {"duration": 3, "energy": 180, "window": list(range(1, 25))},
            "warehouse_hvac_2": {"duration": 3, "energy": 180, "window": list(range(1, 25))},
            "warehouse_hvac_3": {"duration": 3, "energy": 180, "window": list(range(1, 25))},
            "crane_maintenance": {"duration": 3, "energy": 200, "window": list(range(1, 25))},
            "pumping_systems": {"duration": 3, "energy": 240, "window": list(range(1, 25))},
            "ventilation_main": {"duration": 4, "energy": 190, "window": list(range(1, 25))},
            "gate_operations": {"duration": 2, "energy": 110, "window": list(range(1, 25))},
            "security_systems": {"duration": 5, "energy": 150, "window": list(range(1, 25))},
            "admin_buildings": {"duration": 8, "energy": 160, "window": list(range(1, 25))},
            "warehouse_lighting": {"duration": 8, "energy": 120, "window": list(range(1, 25))},
            "yard_lighting": {"duration": 10, "energy": 140, "window": list(range(1, 25))},
            "office_equipment": {"duration": 10, "energy": 150, "window": list(range(1, 25))},
            "emergency_systems": {"duration": 1, "energy": 80, "window": list(range(1, 25))},
            "it_servers": {"duration": 12, "energy": 95, "window": list(range(1, 25))},
            "communication_systems": {"duration": 12, "energy": 85, "window": list(range(1, 25))},
            "monitoring_systems": {"duration": 24, "energy": 75, "window": list(range(1, 25))},
            "backup_systems": {"duration": 2, "energy": 90, "window": list(range(1, 25))},
            "control_systems": {"duration": 24, "energy": 70, "window": list(range(1, 25))},
        }

        # Electricity prices are in dollars per kWh.
        self.prices = {
            t: 0.15 if 6 <= t <= 18 else 0.08
            for t in self.T
        }

        # Baseline demand is in MW.
        self.base_demand = {
            1: 8, 2: 7, 3: 6, 4: 7, 5: 9, 6: 12, 7: 15, 8: 18,
            9: 20, 10: 22, 11: 21, 12: 19, 13: 20, 14: 22, 15: 21,
            16: 19, 17: 17, 18: 15, 19: 13, 20: 11, 21: 10, 22: 9,
            23: 8, 24: 7,
        }

        # The demand charge is in dollars per kW of maximum hourly demand.
        self.demand_charge = 18

        # Storage data are part of the instance but are not decision variables
        # in the load-shifting model implemented in this notebook.
        self.storage = {
            "capacity": 800,     # kWh
            "rate": 150,         # kW charge/discharge rate
            "efficiency": 0.90,  # Round-trip efficiency
        }


mfo_problem = MFOEnergyProblem()
print(
    f"MFO problem initialized with {len(mfo_problem.operations)} operations "
    f"and {len(mfo_problem.T)} time periods."
)

high_energy = sum(
    operation["energy"] > 300 for operation in mfo_problem.operations.values()
)
medium_energy = sum(
    100 <= operation["energy"] <= 300
    for operation in mfo_problem.operations.values()
)
low_energy = sum(
    operation["energy"] < 100 for operation in mfo_problem.operations.values()
)

total_energy = sum(
    operation["energy"] for operation in mfo_problem.operations.values()
)
print(
    "Operations by total-energy category (kWh): "
    f"High: {high_energy}, Medium: {medium_energy}, Low: {low_energy}"
)
print(f"Total operation energy: {total_energy:,} kWh")

In [ ]:
# Implement Moth-Flame Optimization for the encoded operation start times.
class MFOOptimizer:
    def __init__(
        self,
        problem,
        population_size=50,
        max_iterations=1000,
        spiral_constant=1.0,
    ):
        if population_size < 1:
            raise ValueError("population_size must be at least 1.")
        if max_iterations < 1:
            raise ValueError("max_iterations must be at least 1.")

        self.problem = problem
        self.population_size = population_size
        self.max_iterations = max_iterations
        self.spiral_constant = spiral_constant
        self.best_cost_history = []
        self.average_cost_history = []
        self.diversity_history = []
        self.best_solution = None
        self.best_fitness = -np.inf

        self.operation_names = list(problem.operations.keys())
        self.num_operations = len(self.operation_names)

    def encode_solution(self, start_times):
        """Map integer start times to normalized coordinates in [0, 1]."""
        encoded = []
        for op_name in self.operation_names:
            operation = self.problem.operations[op_name]
            max_start = 24 - operation["duration"] + 1
            start_time = start_times[op_name]
            if not 1 <= start_time <= max_start:
                raise ValueError(f"Invalid start time for {op_name}: {start_time}")
            encoded.append((start_time - 1) / 23)
        return np.array(encoded, dtype=float)

    def decode_solution(self, encoded_vector):
        """Round normalized coordinates to feasible integer start times."""
        vector = np.asarray(encoded_vector, dtype=float)
        if vector.shape != (self.num_operations,):
            raise ValueError("The encoded vector must have one coordinate per operation.")

        start_times = {}
        for i, op_name in enumerate(self.operation_names):
            continuous_time = vector[i] * 23 + 1
            start_time = int(np.clip(np.rint(continuous_time), 1, 24))
            operation = self.problem.operations[op_name]
            max_start = 24 - operation["duration"] + 1
            start_times[op_name] = min(max(start_time, 1), max_start)
        return start_times

    def calculate_hourly_demand(self, start_times):
        """Return the hourly demand profile in MW for a feasible schedule."""
        if set(start_times) != set(self.operation_names):
            raise ValueError("A start time must be supplied for every operation.")

        hourly_demand = {}
        for hour in self.problem.T:
            demand = self.problem.base_demand[hour]
            for op_name, start_time in start_times.items():
                operation = self.problem.operations[op_name]
                duration = operation["duration"]
                max_start = 24 - duration + 1
                if not 1 <= start_time <= max_start:
                    raise ValueError(f"Invalid start time for {op_name}: {start_time}")
                hourly_power_kw = operation["energy"] / duration
                if start_time <= hour <= start_time + duration - 1:
                    demand += hourly_power_kw / 1000
            hourly_demand[hour] = demand
        return hourly_demand

    def calculate_solution_cost(self, start_times):
        """Evaluate energy charges and the maximum-demand charge in dollars."""
        hourly_demand = self.calculate_hourly_demand(start_times)
        energy_cost = sum(
            self.problem.prices[hour] * hourly_demand[hour] * 1000
            for hour in self.problem.T
        )
        peak_demand_kw = max(hourly_demand.values()) * 1000
        demand_cost = self.problem.demand_charge * peak_demand_kw
        return energy_cost + demand_cost

    def calculate_fitness(self, encoded_vector):
        """Convert schedule cost to a positive fitness to be maximized."""
        start_times = self.decode_solution(encoded_vector)
        cost = self.calculate_solution_cost(start_times)
        return 1.0 / (1.0 + cost)

    def initialize_population(self):
        """Create random feasible schedules and encode them as moth positions."""
        population = []
        for _ in range(self.population_size):
            start_times = {}
            for op_name in self.operation_names:
                operation = self.problem.operations[op_name]
                max_start = 24 - operation["duration"] + 1
                start_times[op_name] = random.randint(1, max_start)
            population.append(self.encode_solution(start_times))
        return np.asarray(population)

    def spiral_function(
        self, moth_position, flame_position, distance, iteration, max_iterations
    ):
        """Move a moth around its flame using the MFO spiral equation."""
        t = (iteration / max_iterations) * 2 - 1
        spiral_movement = (
            distance
            * np.exp(self.spiral_constant * t)
            * np.cos(2 * np.pi * t)
        )
        return spiral_movement + flame_position

    def calculate_diversity(self, population):
        """Return the mean pairwise Euclidean distance between moths."""
        population = np.asarray(population, dtype=float)
        if len(population) <= 1:
            return 0.0

        # Vectorization preserves the mean pairwise-distance measure while
        # avoiding a separate distance-function call for every moth pair.
        differences = population[:, np.newaxis, :] - population[np.newaxis, :, :]
        distances = np.linalg.norm(differences, axis=2)
        upper_triangle = distances[np.triu_indices(len(population), k=1)]
        return float(np.mean(upper_triangle))

    def update_flames(self, population, iteration):
        """Rank moths and retain the iteration's leading candidates as flames."""
        fitness_values = np.array(
            [self.calculate_fitness(moth) for moth in population]
        )
        sorted_indices = np.argsort(fitness_values)[::-1]
        flame_number = max(
            1,
            int(self.population_size * (1 - iteration / self.max_iterations)),
        )
        flame_number = min(flame_number, len(population))
        flames = population[sorted_indices[:flame_number]].copy()
        return flames, float(fitness_values[sorted_indices[0]])

    def run_mfo(self):
        """Run the population initialization, flame selection, and spiral updates."""
        print(
            f"Running MFO with {self.population_size} moths and "
            f"{self.max_iterations} iterations..."
        )
        moths = self.initialize_population()
        self.best_cost_history = []
        self.average_cost_history = []
        self.diversity_history = []
        self.best_solution = None
        self.best_fitness = -np.inf

        for iteration in range(self.max_iterations):
            flames, current_best_fitness = self.update_flames(moths, iteration)
            if current_best_fitness > self.best_fitness:
                self.best_fitness = current_best_fitness
                self.best_solution = flames[0].copy()

            for i in range(len(moths)):
                flame_index = min(i, len(flames) - 1)
                flame_position = flames[flame_index]
                # MFO uses a coordinate-wise distance so each operation's
                # start-time coordinate follows its own spiral displacement.
                distance = np.abs(flame_position - moths[i])
                moths[i] = self.spiral_function(
                    moths[i],
                    flame_position,
                    distance,
                    iteration,
                    self.max_iterations,
                )
                moths[i] = np.clip(moths[i], 0, 1)

            current_costs = [
                self.calculate_solution_cost(self.decode_solution(moth))
                for moth in moths
            ]
            diversity = self.calculate_diversity(moths)
            best_cost = 1.0 / self.best_fitness - 1.0
            self.best_cost_history.append(best_cost)
            self.average_cost_history.append(float(np.mean(current_costs)))
            self.diversity_history.append(diversity)

            if (iteration + 1) % 100 == 0:
                print(
                    f"Iteration {iteration + 1}: best cost = ${best_cost:,.2f}, "
                    f"population diversity = {diversity:.4f}"
                )

        final_cost = 1.0 / self.best_fitness - 1.0
        print("\nMFO run completed.")
        print(f"Best schedule cost: ${final_cost:,.2f}")
        return self.decode_solution(self.best_solution), final_cost


mfo_optimizer = MFOOptimizer(
    mfo_problem,
    population_size=50,
    max_iterations=1000,
    spiral_constant=1.0,
)
print("MFO optimizer initialized.")

In [ ]:
# Run the optimizer and display its best integer schedule.
start_clock = time.time()
best_solution, best_cost = mfo_optimizer.run_mfo()
execution_seconds = time.time() - start_clock

print(f"\nExecution time: {execution_seconds:.2f} seconds")
print(f"Best schedule cost: ${best_cost:,.2f}")

sorted_operations = sorted(best_solution.items(), key=lambda item: item[1])
operations_by_category = {"High": [], "Medium": [], "Low": []}

for op_name, start_time in sorted_operations:
    operation = mfo_problem.operations[op_name]
    end_time = start_time + operation["duration"] - 1
    if operation["energy"] > 300:
        category = "High"
    elif operation["energy"] >= 100:
        category = "Medium"
    else:
        category = "Low"

    start_tariff = "Peak-price" if mfo_problem.prices[start_time] == 0.15 else "Off-peak-price"
    operation_info = (
        f"{op_name.replace('_', ' ').title():<30}: "
        f"Hours {start_time:2d}-{end_time:2d} "
        f"({operation['duration']} h, {operation['energy']} kWh); "
        f"start-hour tariff: {start_tariff}"
    )
    operations_by_category[category].append(operation_info)

for category, heading in [
    ("High", "High-energy operations (>300 kWh)"),
    ("Medium", "Medium-energy operations (100-300 kWh)"),
    ("Low", "Low-energy operations (<100 kWh)"),
]:
    print(f"\n{heading}:")
    for operation_info in operations_by_category[category]:
        print(f"  {operation_info}")

In [ ]:
def calculate_baseline_for_mfo(problem, optimizer):
    """Evaluate the feasible baseline schedule with every operation at hour 1."""
    baseline_solution = {op_name: 1 for op_name in problem.operations}
    baseline_cost = optimizer.calculate_solution_cost(baseline_solution)
    return baseline_solution, baseline_cost


def calculate_peak_demand_mfo(problem, optimizer, solution):
    """Return the maximum hourly demand, in MW, for a schedule."""
    hourly_demand = optimizer.calculate_hourly_demand(solution)
    return max(hourly_demand.values())


def analyze_peak_off_peak_scheduling(problem, solution):
    """Measure operation starts and energy by the tariff applying in each hour."""
    peak_start_operations = 0
    off_peak_start_operations = 0
    peak_energy = 0.0
    off_peak_energy = 0.0

    for op_name, start_time in solution.items():
        operation = problem.operations[op_name]
        duration = operation["duration"]
        energy_per_hour = operation["energy"] / duration
        if problem.prices[start_time] == 0.15:
            peak_start_operations += 1
        else:
            off_peak_start_operations += 1

        for hour in range(start_time, start_time + duration):
            if problem.prices[hour] == 0.15:
                peak_energy += energy_per_hour
            else:
                off_peak_energy += energy_per_hour

    return {
        "peak_start_operations": peak_start_operations,
        "off_peak_start_operations": off_peak_start_operations,
        "peak_energy": peak_energy,
        "off_peak_energy": off_peak_energy,
    }


baseline_solution, baseline_cost = calculate_baseline_for_mfo(
    mfo_problem, mfo_optimizer
)
baseline_peak = calculate_peak_demand_mfo(
    mfo_problem, mfo_optimizer, baseline_solution
)
mfo_peak = calculate_peak_demand_mfo(
    mfo_problem, mfo_optimizer, best_solution
)

cost_savings = baseline_cost - best_cost
cost_savings_percent = 100 * cost_savings / baseline_cost
peak_reduction = baseline_peak - mfo_peak
peak_reduction_percent = 100 * peak_reduction / baseline_peak

print("\n=== COST COMPARISON ===")
print(f"Baseline cost:       ${baseline_cost:,.2f}")
print(f"MFO schedule cost:   ${best_cost:,.2f}")
print(f"Cost savings:        ${cost_savings:,.2f} ({cost_savings_percent:.1f}%)")
print("\n=== PEAK DEMAND COMPARISON ===")
print(f"Baseline peak:       {baseline_peak:.3f} MW")
print(f"MFO schedule peak:   {mfo_peak:.3f} MW")
print(f"Peak reduction:      {peak_reduction:.3f} MW ({peak_reduction_percent:.1f}%)")

baseline_schedule = analyze_peak_off_peak_scheduling(
    mfo_problem, baseline_solution
)
mfo_schedule = analyze_peak_off_peak_scheduling(mfo_problem, best_solution)
print("\n=== TARIFF-PERIOD SCHEDULING ANALYSIS ===")
print(
    "Baseline operation starts: "
    f"{baseline_schedule['peak_start_operations']} during peak-price hours, "
    f"{baseline_schedule['off_peak_start_operations']} during off-peak-price hours"
)
print(
    "Baseline energy by operating hour: "
    f"{baseline_schedule['peak_energy']:,.1f} kWh peak-price, "
    f"{baseline_schedule['off_peak_energy']:,.1f} kWh off-peak-price"
)
print(
    "MFO operation starts: "
    f"{mfo_schedule['peak_start_operations']} during peak-price hours, "
    f"{mfo_schedule['off_peak_start_operations']} during off-peak-price hours"
)
print(
    "MFO energy by operating hour: "
    f"{mfo_schedule['peak_energy']:,.1f} kWh peak-price, "
    f"{mfo_schedule['off_peak_energy']:,.1f} kWh off-peak-price"
)
print(
    "Change in off-peak-period energy: "
    f"{mfo_schedule['off_peak_energy'] - baseline_schedule['off_peak_energy']:,.1f} kWh"
)

assert set(best_solution) == set(mfo_problem.operations)
assert all(
    1 <= best_solution[op_name] <= 25 - operation["duration"]
    for op_name, operation in mfo_problem.operations.items()
)
assert np.isclose(
    mfo_optimizer.calculate_solution_cost(best_solution), best_cost
)

In [ ]:
def create_mfo_visualizations(
    problem,
    optimizer,
    baseline_solution,
    baseline_cost,
    baseline_peak,
    mfo_peak,
):
    """Plot search progress, diversity, demand, and baseline-relative metrics."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Moth-Flame Optimization: Energy Load-Shifting Results", fontsize=16)

    generations = np.arange(1, len(optimizer.best_cost_history) + 1)

    ax1 = axes[0, 0]
    ax1.plot(generations, optimizer.best_cost_history, label="Best cost so far")
    ax1.plot(
        generations,
        optimizer.average_cost_history,
        linestyle="--",
        alpha=0.8,
        label="Current-population average cost",
    )
    ax1.set_xlabel("Iteration")
    ax1.set_ylabel("Daily cost ($)")
    ax1.set_title("Cost During the MFO Search")
    ax1.set_yscale("log")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2 = axes[0, 1]
    ax2.plot(generations, optimizer.diversity_history, color="purple")
    ax2.set_xlabel("Iteration")
    ax2.set_ylabel("Mean pairwise Euclidean distance")
    ax2.set_title("Population Diversity")
    ax2.grid(True, alpha=0.3)

    ax3 = axes[1, 0]
    hours = list(problem.T)
    baseline_profile = optimizer.calculate_hourly_demand(baseline_solution)
    mfo_solution = optimizer.decode_solution(optimizer.best_solution)
    mfo_profile = optimizer.calculate_hourly_demand(mfo_solution)
    baseline_values = [baseline_profile[hour] for hour in hours]
    mfo_values = [mfo_profile[hour] for hour in hours]
    ax3.plot(hours, baseline_values, marker="o", markersize=3, label="Baseline")
    ax3.plot(hours, mfo_values, marker="s", markersize=3, label="MFO schedule")
    ax3.set_xlabel("Hour")
    ax3.set_ylabel("Power demand (MW)")
    ax3.set_title("Hourly Power-Demand Profile")
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    ax4 = axes[1, 1]
    metric_names = ["Daily cost", "Peak demand"]
    baseline_values = [100.0, 100.0]
    mfo_values = [
        100 * optimizer.best_cost_history[-1] / baseline_cost,
        100 * mfo_peak / baseline_peak,
    ]
    positions = np.arange(len(metric_names))
    width = 0.35
    ax4.bar(positions - width / 2, baseline_values, width, label="Baseline")
    ax4.bar(positions + width / 2, mfo_values, width, label="MFO schedule")
    ax4.set_xticks(positions)
    ax4.set_xticklabels(metric_names)
    ax4.set_ylabel("Index (baseline = 100)")
    ax4.set_title("Cost and Peak Demand Relative to Baseline")
    ax4.legend()
    ax4.grid(True, axis="y", alpha=0.3)

    fig.tight_layout()
    plt.show()

    print("\n=== SEARCH-PROGRESS SUMMARY ===")
    print(f"Initial best cost: ${optimizer.best_cost_history[0]:,.2f}")
    print(f"Final best cost:   ${optimizer.best_cost_history[-1]:,.2f}")
    print(f"Average diversity: {np.mean(optimizer.diversity_history):.4f}")
    print(f"Final diversity:   {optimizer.diversity_history[-1]:.4f}")

    # Use a 100-iteration window to describe a sustained small change in
    # the non-increasing best-so-far cost; this is a diagnostic, not a proof
    # that the search has reached a global optimum.
    convergence_index = None
    for i in range(100, len(optimizer.best_cost_history)):
        reference_cost = optimizer.best_cost_history[i - 100]
        relative_improvement = (
            reference_cost - optimizer.best_cost_history[i]
        ) / reference_cost
        if relative_improvement < 0.01:
            convergence_index = i
            break

    if convergence_index is None:
        print(
            "No 100-iteration window with less than 1% improvement was detected."
        )
    else:
        print(
            "The first 100-iteration window with less than 1% improvement "
            f"ends at iteration {convergence_index + 1}."
        )


create_mfo_visualizations(
    mfo_problem,
    mfo_optimizer,
    baseline_solution,
    baseline_cost,
    baseline_peak,
    mfo_peak,
)

In [ ]:
def analyze_spiral_behavior(problem, optimizer):
    """Illustrate sampled spiral updates for one moth in a diagnostic run."""
    print("\n=== SPIRAL-MOVEMENT ANALYSIS ===")
    sample_moth_index = 0
    sample_moth_positions = []
    sampled_iterations = list(range(0, min(200, optimizer.max_iterations), 10))

    # This diagnostic creates a separate population. Its trajectory illustrates
    # sampled MFO updates and is not the stored trajectory of the main run.
    moths = optimizer.initialize_population()
    for iteration in sampled_iterations:
        flames, _ = optimizer.update_flames(moths, iteration)
        flame_index = min(sample_moth_index, len(flames) - 1)
        flame_position = flames[flame_index]
        moth_position = moths[sample_moth_index]
        sample_moth_positions.append(moth_position.copy())

        distance = np.abs(flame_position - moth_position)
        moths[sample_moth_index] = optimizer.spiral_function(
            moth_position,
            flame_position,
            distance,
            iteration,
            optimizer.max_iterations,
        )
        moths[sample_moth_index] = np.clip(moths[sample_moth_index], 0, 1)

    positions = np.asarray(sample_moth_positions)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    if len(positions) > 1:
        ax1.plot(positions[:, 0], positions[:, 1], marker="o", label="Sampled moth path")
        ax1.scatter(positions[0, 0], positions[0, 1], color="green", s=80, label="Start")
        ax1.scatter(positions[-1, 0], positions[-1, 1], color="red", s=80, label="End")
        ax1.set_xlabel("Normalized coordinate 1 (operation start time)")
        ax1.set_ylabel("Normalized coordinate 2 (operation start time)")
        ax1.set_title("Spiral Updates: Two-Coordinate Projection")
        ax1.legend()
        ax1.grid(True, alpha=0.3)

    distances_from_best = [
        euclidean(position, optimizer.best_solution) for position in positions
    ]
    ax2.plot(sampled_iterations, distances_from_best, marker="s", color="red")
    ax2.set_xlabel("MFO iteration sampled")
    ax2.set_ylabel("Euclidean distance from best encoded solution")
    ax2.set_title("Sampled Moth Distance from the Run's Best Solution")
    ax2.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()
    print(f"Recorded {len(sample_moth_positions)} sampled moth positions.")


analyze_spiral_behavior(mfo_problem, mfo_optimizer)

## Interpreting the method and its results

MFO is useful here as a population-based search procedure: multiple candidate schedules are updated relative to ranked candidate flames, while a decreasing flame count changes how many leading candidates guide the search. The spiral update and population diversity are properties of this search process; neither guarantees that a global optimum will be found or that a moth will escape every local optimum.

The cost history reports the best schedule encountered so far and the mean cost of the current population. These are different summaries: the best-so-far cost is non-increasing during a run, while the current-population average may rise or fall after moth movements. Diversity is the mean pairwise Euclidean distance between normalized encoded schedules, so it describes separation in the search representation rather than a direct measure of operational performance.

The baseline comparison uses the feasible schedule in which every operation starts at hour 1. The reported savings are relative to that baseline, not a proof of optimality. Peak and off-peak energy are assigned hour by hour, which accounts for operations that span tariff boundaries. The visual performance chart indexes daily cost and peak demand separately to their baseline values because their physical units differ.

### Strengths and limitations

- Population-based updates can explore several candidate schedules in one run.
- The continuous encoding allows the spiral movement equation to be used, while decoding enforces integer start times and the daily horizon.
- The result depends on the population, iteration limit, spiral constant, and random initialization. A single run does not establish solution quality for every instance.
- This implementation does not include precedence, shared-resource constraints, storage dispatch, or an optimality certificate.
- Cost evaluations and pairwise diversity calculations can become computationally expensive as the population or number of operations grows.

### Scope for further study

Natural extensions within the same scheduling setting include adding realistic operation windows or additional operational constraints, and examining how the MFO search behaves on larger instances. Such extensions require corresponding changes to the model and feasibility handling; they are not part of the instance solved here.